data exploration

In [1]:
import pandas as pd

# 1. Load the Master Clauses CSV
# Note: Replace 'master_clauses.csv' with the actual file path if it's stored in a folder
file_path = 'CUAD_v1/master_clauses.csv'
df = pd.read_csv(file_path)

# 2. Check the overall shape of the dataset
print(f"Dataset Shape: {df.shape[0]} rows and {df.shape[1]} columns.\n")

# 3. Display all column names to identify the specific 41 categories
print("All Columns in the dataset:")
for i, col in enumerate(df.columns):
    print(f"{i}: {col}")

print("\n" + "="*50 + "\n")

# 4. Preview the first 3 rows to see how the text and answers are structured
# Using display() instead of print() makes it render nicely in Jupyter
display(df.head(3))

Dataset Shape: 510 rows and 83 columns.

All Columns in the dataset:
0: Filename
1: Document Name
2: Document Name-Answer
3: Parties
4: Parties-Answer
5: Agreement Date
6: Agreement Date-Answer
7: Effective Date
8: Effective Date-Answer
9: Expiration Date
10: Expiration Date-Answer
11: Renewal Term
12: Renewal Term-Answer
13: Notice Period To Terminate Renewal
14: Notice Period To Terminate Renewal- Answer
15: Governing Law
16: Governing Law-Answer
17: Most Favored Nation
18: Most Favored Nation-Answer
19: Competitive Restriction Exception
20: Competitive Restriction Exception-Answer
21: Non-Compete
22: Non-Compete-Answer
23: Exclusivity
24: Exclusivity-Answer
25: No-Solicit Of Customers
26: No-Solicit Of Customers-Answer
27: No-Solicit Of Employees
28: No-Solicit Of Employees-Answer
29: Non-Disparagement
30: Non-Disparagement-Answer
31: Termination For Convenience
32: Termination For Convenience-Answer
33: Rofr/Rofo/Rofn
34: Rofr/Rofo/Rofn-Answer
35: Change Of Control
36: Change Of Co

,Filename,Document Name,Document Name-Answer,Parties,Parties-Answer,Agreement Date,Agreement Date-Answer,Effective Date,Effective Date-Answer,Expiration Date,...,Liquidated Damages,Liquidated Damages-Answer,Warranty Duration,Warranty Duration-Answer,Insurance,Insurance-Answer,Covenant Not To Sue,Covenant Not To Sue-Answer,Third Party Beneficiary,Third Party Beneficiary-Answer
0,CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605...,['MARKETING AFFILIATE AGREEMENT'],MARKETING AFFILIATE AGREEMENT,"['BIRCH FIRST GLOBAL INVESTMENTS INC.', 'MA', ...","Birch First Global Investments Inc. (""Company""...","['8th day of May 2014', 'May 8, 2014']",5/8/14,['This agreement shall begin upon the date of ...,NaN,['This agreement shall begin upon the date of ...,...,[],No,"[""COMPANY'S SOLE AND EXCLUSIVE LIABILITY FOR T...",Yes,[],No,[],No,[],No
1,EuromediaHoldingsCorp_20070215_10SB12G_EX-10.B...,['VIDEO-ON-DEMAND CONTENT LICENSE AGREEMENT'],VIDEO-ON-DEMAND CONTENT LICENSE AGREEMENT,"['EuroMedia Holdings Corp.', 'Rogers', 'Rogers...","Rogers Cable Communications Inc. (""Rogers""); E...","['July 11 , 2006']",7/11/06,"['July 11 , 2006']",7/11/06,"['The term of this Agreement (the ""Initial Ter...",...,[],No,[],No,[],No,[],No,[],No
2,FulucaiProductionsLtd_20131223_10-Q_EX-10.9_83...,['CONTENT DISTRIBUTION AND LICENSE AGREEMENT'],CONTENT DISTRIBUTION AND LICENSE AGREEMENT,"['Producer', 'Fulucai Productions Ltd.', 'Conv...","CONVERGTV, INC. (“ConvergTV”); Fulucai Product...","['November 15, 2012']",11/15/12,"['November 15, 2012']",11/15/12,[],...,[],No,[],No,[],No,[],No,[],No


filtering data based in clauses related to CS freelancers

In [2]:
# 1. Define the specific categories relevant to CS Freelance Contracts
cs_categories = [
    "Ip Ownership Assignment",
    "Non-Compete",
    "Exclusivity",
    "No-Solicit Of Customers",
    "Cap On Liability",
    "License Grant",
    "Source Code Escrow",
    "Minimum Commitment",
    "Non-Transferable License",
    "Third Party Beneficiary"
]

# 2. Initialize an empty list to hold our clean, structured data
processed_clauses = []

# 3. Loop through each category to extract the relevant text
for category in cs_categories:
    answer_col = f"{category}-Answer"
    
    # Filter for rows where the contract actually contains this clause ("Yes")
    # We use .str.strip().str.title() to catch messy formatting like " yes " or "YES"
    if answer_col in df.columns:
        valid_rows = df[df[answer_col].astype(str).str.strip().str.title() == "Yes"]
        
        # Extract the text and build our metadata dictionaries
        for index, row in valid_rows.iterrows():
            text_chunk = str(row[category])
            
            # Skip any empty or corrupted rows
            if text_chunk == "nan" or not text_chunk.strip():
                continue
                
            # Clean up the specific dataset quirks (like replacing <omitted> with standard ellipses)
            clean_text = text_chunk.replace("<omitted>", " [...] ")
            
            # Create the dictionary format required for Pinecone metadata
            processed_clauses.append({
                "id": f"{row['Filename']}_{category.replace(' ', '')}_{index}", # Unique ID
                "text": clean_text,                                             # The field we will embed
                "clause_type": category,                                        # Metadata: Type of clause
                "source_file": row['Filename']                                  # Metadata: Original file
            })

# 4. Convert our processed list back into a DataFrame to preview it
processed_df = pd.DataFrame(processed_clauses)

print(f"Successfully extracted {len(processed_df)} highly relevant CS clauses!")
print("\nPreview of the structured data ready for Pinecone:")
display(processed_df.head())

Successfully extracted 1336 highly relevant CS clauses!

Preview of the structured data ready for Pinecone:


,id,text,clause_type,source_file
0,MusclepharmCorp_20170208_10-KA_EX-10.38_989358...,"[""All such works based upon the Trademarks and...",Ip Ownership Assignment,MusclepharmCorp_20170208_10-KA_EX-10.38_989358...
1,TomOnlineInc_20060501_20-F_EX-4.46_749700_EX-4...,['if such rights comprise (i) intellectual pro...,Ip Ownership Assignment,TomOnlineInc_20060501_20-F_EX-4.46_749700_EX-4...
2,ConformisInc_20191101_10-Q_EX-10.6_11861402_EX...,"[""Conformis agrees to assign and hereby assi...",Ip Ownership Assignment,ConformisInc_20191101_10-Q_EX-10.6_11861402_EX...
3,FuelcellEnergyInc_20191106_8-K_EX-10.1_1186800...,"['FCE will assign, and hereby assigns, to Exxo...",Ip Ownership Assignment,FuelcellEnergyInc_20191106_8-K_EX-10.1_1186800...
4,ReedsInc_20191113_10-Q_EX-10.4_11888303_EX-10....,"[""Reed's will exclusively own all Deliverables...",Ip Ownership Assignment,ReedsInc_20191113_10-Q_EX-10.4_11888303_EX-10....


pushing the documents to pinecone database

In [3]:
import os
from dotenv import load_dotenv
import time 
from pinecone import Pinecone, ServerlessSpec
from tqdm.auto import tqdm 

#loading environment variables from .env file
load_dotenv()

# 1. Initialize Pinecone connection
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
pc = Pinecone(api_key=PINECONE_API_KEY)
index_name = "contracts"

# 2. Check if the index exists. If not, create it
if index_name not in pc.list_indexes().names():
    print(f"Creating index '{index_name}'...")
    pc.create_index(
        name=index_name,
        dimension=1024, # Llama-text-embed-v2 outputs 1024 dimensions
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws", 
            region="us-east-1" 
        )
    )
    
    # Wait until the index is fully spun up before proceeding
    print("Waiting for index to initialize...")
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1)
    print("Index created and ready!")

# Connect to the index
index = pc.Index(index_name)

# 3. Batch process and upload
batch_size = 50 
print(f"Starting upload of {len(processed_df)} clauses to Pinecone...")

for i in tqdm(range(0, len(processed_df), batch_size)):
    # Get the current batch of rows
    batch = processed_df.iloc[i:i+batch_size]
    texts = batch['text'].tolist()
    
    try:
        # 4. Generate Embeddings using Pinecone's Inference API
        embeddings_response = pc.inference.embed(
            model="llama-text-embed-v2",
            inputs=texts,
            parameters={"input_type": "passage", "truncate": "END"}
        )
        
        # 5. Prepare the data for upload
        vectors = []
        for j, (_, row) in enumerate(batch.iterrows()):
            vectors.append({
                "id": str(row['id']), 
                
                # THE CRITICAL FIX: Must access .data[j]
                "values": embeddings_response.data[j].values, 
                
                "metadata": {
                    "text": str(row['text']),
                    "clause_type": str(row['clause_type']),
                    "source_file": str(row['source_file'])
                }
            })
            
        # 6. Upsert (Upload) the structured batch
        index.upsert(vectors=vectors)
        
        # 7. SAFETY: Pause for 2 seconds to avoid Rate Limit (429 Too Many Requests)
        time.sleep(2)
        
    except Exception as e:
        print(f"\nUpload stopped due to error at batch {i}. Error details: {e}")
        print("Wait a minute for rate limits to reset, then resume from this batch.")
        break

print("\nProcess finished.")

c:\UNI\FAST BS(CS)\8th Semester\AAI\Automated-Contract-Risk-Analysis\aai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Starting upload of 1336 clauses to Pinecone...


 30%|██▉       | 8/27 [00:47<01:53,  5.96s/it]


Upload stopped due to error at batch 400. Error details: (400)
Reason: Bad Request
HTTP response headers: HTTPHeaderDict({'Date': 'Sat, 18 Apr 2026 22:13:09 GMT', 'Content-Type': 'application/json', 'Content-Length': '151', 'Connection': 'keep-alive', 'x-pinecone-request-latency-ms': '390', 'x-envoy-upstream-service-time': '4', 'x-pinecone-response-duration-ms': '392', 'server': 'envoy'})
HTTP response body: {"code":3,"message":"Vector ID must be ASCII, but got 'LECLANCHÉ S.A. - JOINT DEVELOPMENT AND MARKETING AGREEMENT.PDF_Exclusivity_470'","details":[]}

Wait a minute for rate limits to reset, then resume from this batch.

Process finished.


In [3]:
import os
from dotenv import load_dotenv
import time 
from pinecone import Pinecone
from tqdm.auto import tqdm 

#loading environment variables from .env file
load_dotenv()

# 1. Initialize Pinecone connection
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
pc = Pinecone(api_key=PINECONE_API_KEY)
index_name = "contracts"
index = pc.Index(index_name)

# 2. Dynamic Resume Configuration
# We ask Pinecone exactly how many vectors it already has!
stats = index.describe_index_stats()
start_index = stats.total_vector_count 
batch_size = 50 

print(f"Pinecone already has {start_index} clauses saved.")
if start_index >= len(processed_df):
    print("Upload is already 100% complete!")
else:
    print(f"Resuming upload from row {start_index} out of {len(processed_df)}...")

    for i in tqdm(range(start_index, len(processed_df), batch_size)):
        batch = processed_df.iloc[i:i+batch_size]
        texts = batch['text'].tolist()
        
        try:
            # 3. Generate Embeddings 
            embeddings_response = pc.inference.embed(
                model="llama-text-embed-v2",
                inputs=texts,
                parameters={"input_type": "passage", "truncate": "END"}
            )
            
            # 4. Prepare the data for upload (With the ASCII safety fix)
            vectors = []
            for j, (_, row) in enumerate(batch.iterrows()):
                raw_id = str(row['id'])
                safe_id = raw_id.encode('ascii', 'ignore').decode('ascii')
                
                vectors.append({
                    "id": safe_id, 
                    "values": embeddings_response.data[j].values, 
                    "metadata": {
                        "text": str(row['text']),
                        "clause_type": str(row['clause_type']),
                        "source_file": str(row['source_file'])
                    }
                })
                
            # 5. Upsert (Upload) the structured batch
            index.upsert(vectors=vectors)
            
            # 6. Safety Pause
            time.sleep(2)
            
        except Exception as e:
            print(f"\nUpload stopped due to error at batch {i}. Error details: {e}")
            break

    print("\nProcess finished. Your Pinecone vector database is complete!")

c:\UNI\FAST BS(CS)\8th Semester\AAI\Automated-Contract-Risk-Analysis\aai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Pinecone already has 400 clauses saved.
Resuming upload from row 400 out of 1336...


100%|██████████| 19/19 [02:13<00:00,  7.01s/it]


Process finished. Your Pinecone vector database is complete!
